In [ ]:
import numpy as np
import pandas as pd 
import math

In [ ]:
def LRC_f(phenotypes, genotypes, expression, g_s_p, phen):
    bxd = phenotypes.filter(like='BXD').columns.intersection(expression.columns.intersection(genotypes.loc[g_s_p['snp']].dropna().index))
    genotypes = genotypes.loc[g_s_p['snp']][bxd]
    expression = expression.loc[g_s_p['gene']][bxd]
    phenotypes = phenotypes.loc[phen].filter(like='BXD')[bxd]
    LRC = pd.DataFrame({'L': genotypes, 'R': expression, 'C': phenotypes})
    return LRC


def p_cond(b_value, mean, var):
    x = math.log(1 / (math.sqrt(2 * math.pi * var)))
    y = -((b_value - mean) ** 2) / (2 * var)
    return x+y


def data_for_p_cond(LRC, loc, x, y):
    mean_x = LRC[x].mean()
    var_x = LRC[x].var()
    mean_y = LRC[y].mean()
    var_y = LRC[y].var()
    corr = int(LRC[y].corr(LRC[x]))
    x = LRC.loc[loc][x]
    conditional_rc_prob_mean = mean_y + ((corr*np.sqrt(var_y))/np.sqrt(var_x))*(x-mean_x)
    conditional_rc_prob_var = var_y*(1-np.square(corr))
    return conditional_rc_prob_mean, conditional_rc_prob_var


def likeli(LRC, phen, name, log):
    mean_l_r = LRC.groupby('L').mean().loc[phen, 'R']
    var_l_r = LRC.groupby('L').var().loc[phen, 'R']
    mean_c_l = LRC.groupby('L').mean().loc[phen, 'C']
    var_c_l = LRC.groupby('L').var().loc[phen, 'C']
    mean_r_c, var_r_c = data_for_p_cond(LRC, name, x='R', y='C')
    mean_cr, var_cr = data_for_p_cond(LRC, name, x='C', y='R')
    
    mod_1 = log + p_cond(b_value=LRC.loc[name]['R'], mean=mean_l_r, var=var_l_r) + p_cond(b_value=LRC.loc[name]['C'], mean=mean_r_c, var=var_r_c)
    mod_2 = log + (p_cond(b_value=LRC.loc[name]['C'], mean=mean_c_l, var=var_c_l)) + p_cond(b_value=LRC.loc[name]['R'], mean=mean_cr, var=var_cr)
    mod_3 = log + p_cond(b_value=LRC.loc[name]['R'], mean=mean_l_r, var=var_l_r) + p_cond(b_value=LRC.loc[name]['C'], mean=mean_c_l, var=var_c_l)
    return mod_1, mod_2, mod_3

In [ ]:
genotypes_myeloid = pd.read_excel('preprocess_sub/myeloid_pos_preprocessed.xlsx', index_col=0)
genotypes_liver = pd.read_excel('preprocess_sub/liver_pos_preprocessed.xlsx', index_col=0)
myeloid_exp = pd.read_excel('preprocess_sub/myeloid_exp_preprocessed.xlsx', index_col=0)
liver_exp = pd.read_excel('preprocess_sub/liver_exp_preprocessed.xlsx', index_col=0)

phenotypes = pd.read_excel('preprocess_sub/phenotypes_preprocessed.xlsx',index_col=0)
comb_eQTL_QTL = pd.read_excel('comb_eQTL_liver_QTL_1003.xlsx').drop('Unnamed: 0', axis=1)
comb_eQTL_QTL['phe_num'] = 1003

In [ ]:
comb_eQTL_QTL

In [ ]:
phenotypes

In [ ]:
liver_gen = genotypes_liver.applymap(lambda t: t.upper() if isinstance(t, str) else t)
liver_gen = liver_gen.replace({'B': 0, 'D': 2, 'H': None})  # markings

myeloid_gen = genotypes_myeloid.applymap(lambda t: t.upper() if isinstance(t, str) else t)
myeloid_gen = myeloid_gen.replace({'B': 0, 'D': 2, 'H': None})  # markings

In [ ]:
genotypes = liver_gen
exp = liver_exp

In [ ]:
log_value = np.log(0.5)

In [ ]:
mod1_dict = dict()
mod2_dict = dict()
mod3_dict = dict()
for i in range(0,comb_eQTL_QTL.shape[0]):
    mod_1_score = 0
    mod_2_score = 0
    mod_3_score = 0
    g_s_p = comb_eQTL_QTL.loc[i].loc[['gene', 'snp', 'phe_num']]
    LRC = LRC_f(phenotypes, genotypes, exp, g_s_p, 1003)
    LRC = LRC.astype(float)
    for k in LRC.iterrows():
        name = k[0]
        phe = LRC.loc[name]['L']
        mod_like = likeli(LRC, phe, name, log_value)
        mod_1_score += mod_like[0]
        mod_2_score += mod_like[1]
        mod_3_score += mod_like[2]
    mod1_dict[i] = mod_1_score
    mod2_dict[i] = mod_2_score
    mod3_dict[i] = mod_3_score

In [ ]:
comb_eQTL_QTL['L(M1)'] = comb_eQTL_QTL.index.map(mod1_dict)
comb_eQTL_QTL['L(M2)'] = comb_eQTL_QTL.index.map(mod2_dict)
comb_eQTL_QTL['L(M3)'] = comb_eQTL_QTL.index.map(mod3_dict)
comb_eQTL_QTL

In [ ]:
Likelihood_dict = comb_eQTL_QTL.loc[1][['L(M1)', 'L(M2)', 'L(M3)']].to_dict()
Likelihood_list = list(Likelihood_dict.values())
likeli_i = {i: j for j, i in Likelihood_dict.items()}

In [ ]:
LR_list = list()
best_model = list()
for t in range(0,comb_eQTL_QTL.shape[0]):
    Likelihood_dict = comb_eQTL_QTL.loc[t][['L(M1)', 'L(M2)', 'L(M3)']].to_dict()
    Likelihood_list = list(Likelihood_dict.values())
    Likelihood_list.sort(reverse=True)
    LR_list.append(Likelihood_list[0]-Likelihood_list[1])
    likeli_i = {i: j for j, i in Likelihood_dict.items()}
    best_model.append(likeli_i[Likelihood_list[0]])

In [ ]:
comb_eQTL_QTL['LR'] = LR_list
comb_eQTL_QTL['best_model'] = best_model

In [ ]:
models = dict()
mod1_dict = dict()
mod2_dict = dict()
mod3_dict = dict()
for i in range(0,comb_eQTL_QTL.shape[0]):
    mod_scores = list()
    mod1_score = 0
    mod2_score = 0
    mod3_score = 0
    g_s_p = comb_eQTL_QTL.loc[i].loc[['gene', 'snp', 'phe_num']]
    LRC = LRC_f(phenotypes, genotypes, exp, g_s_p, 1003)
    LRC = LRC.astype(float)
    for j in range(0,10):
        LRC_iter = LRC.copy()
        LRC_iter['R'] = LRC_iter['R'].sample(frac=1).values
        LRC_iter['C'] = LRC_iter['C'].sample(frac=1).values
        for k in LRC_iter.iterrows():
            name = k[0]
            phen = LRC_iter.loc[name]['L']
            mod_like = likeli(LRC_iter, phen, name, log_value)
            mod1_score += mod_like[0]
            mod2_score += mod_like[1]
            mod3_score += mod_like[2]
        mod_scores.append([mod1_score, mod2_score, mod3_score])
    models[i] = mod_scores

In [ ]:
p_values = list()
for i in range(0, len(models)):
    likelihoods = pd.DataFrame(models[0], columns = ['L(M1)', 'L(M2)', 'L(M3)'])
    LR_list = list()
    for j in range(0,likelihoods.shape[0]):
        likelihood_mod = likelihoods.loc[j][['L(M1)', 'L(M2)', 'L(M3)']].tolist()
        likelihood_mod.sort(reverse=True)
        LR_list.append(likelihood_mod[0]-likelihood_mod[1])
    LR = comb_eQTL_QTL.loc[i]['LR']
    LR_list.append(LR)
    LR_list.sort(reverse=True)
    temp_count = 0
    for i in LR_list:
        if LR > i:
            break
        temp_count = temp_count + 1
    p_values.append(temp_count/likelihoods.shape[0])

In [ ]:
comb_eQTL_QTL['p-value'] = p_values

In [ ]:
comb_eQTL_QTL

In [ ]:
comb_eQTL_QTL.to_excel('causality_results_liver.xlsx')